# Memory and GPU — Python demonstrations
Use the same two experiments as the memory slides: compiled loop order, then batching.
Library layout handling, output-buffer reuse and arithmetic intensity are optional references.
The GPU section follows afterwards. Inputs may be C-contiguous, F-contiguous or strided;
inspect the actual array rather than assuming that every NumPy array is row-major.

In [ ]:
import timeit

def best_time(fn, repeat=7):
    """Time per call (s), %timeit-style: auto-scale then min over `repeat` trials."""
    number = 1
    while timeit.timeit(fn, number=number) < 0.05:
        number *= 10
    return min(timeit.repeat(fn, number=number, repeat=repeat)) / number

def ms(t): return f"{t*1e3:.3f} ms"
def us(t): return f"{t*1e6:.2f} µs"


In [ ]:
import numpy as np

## 1. Compare two compiled traversals
Predict first. Numba compiles both explicit loops before measurement; timing therefore does not
mainly measure the Python interpreter. Changing addition order can change floating-point rounding.

In [ ]:
from numba import njit
N = 4000
A = np.random.rand(N, N)  # default C-contiguous layout

@njit
def sum_j_outer(A):
    s = 0.0
    for j in range(A.shape[1]):
        for i in range(A.shape[0]):
            s += A[i, j]
    return s

@njit
def sum_i_outer(A):
    s = 0.0
    for i in range(A.shape[0]):
        for j in range(A.shape[1]):
            s += A[i, j]
    return s

# Compile both loops and validate within tolerance before timing.
assert np.isclose(sum_i_outer(A), sum_j_outer(A))
t_i = best_time(lambda: sum_i_outer(A))
t_j = best_time(lambda: sum_j_outer(A))
print("shape", A.shape, "dtype", A.dtype, "strides", A.strides)
print("i outer, j inner:", ms(t_i), "j outer, i inner:", ms(t_j))
print("ratio:", max(t_i,t_j)/min(t_i,t_j))


A size-dependent difference supports a locality explanation, but does not isolate cache,
prefetching and translation effects. Try a smaller array; loop order need not become irrelevant
just because data fits in cache.

## 2. Batch the same workload
Compare the same 128 examples individually with GEMV, then by batches. Hold BLAS threads and
dtype fixed. Report throughput and the input/output payload per batch; it is not peak RAM.

In [ ]:
import time, platform
import numpy as np
from threadpoolctl import threadpool_limits, threadpool_info
print(platform.python_version(), np.__version__)
threadpool_limits(limits=1, user_api="blas")  # one BLAS thread for ALL timings below
print(threadpool_info())
rng = np.random.default_rng(2026)
m, total = 1024, 128  # increase m only if time/memory allow
W = rng.random((m, m), dtype=np.float32)
X = np.asfortranarray(rng.random((m, total), dtype=np.float32))

def individual():
    # A true matrix-vector product for EACH of the same total examples.
    return np.column_stack([W @ X[:, i] for i in range(total)])

def elapsed(fn):
    fn()  # warm up
    times = []
    for _ in range(3):
        start = time.perf_counter()
        fn()
        times.append(time.perf_counter() - start)
    return np.median(times)

def batched(B):
    ## SOLUTION: multiply W by successive B-column slices of X, concatenate the results ##
    return np.concatenate([W @ X[:, i:i+B] for i in range(0, total, B)], axis=1)
    ## END ##

# The whole table
expected = individual()
reference_time = elapsed(individual)
print("Float32; BLAS threads=1; same", total, "examples; W bytes:", W.nbytes)
print("reference GEMV: seconds", reference_time, "µs/example", reference_time/total*1e6)
print("B | total seconds | µs/example | examples/s | speedup vs GEMV | batch payload KiB")
for B in (1, 8, 64):
    assert np.allclose(batched(B), expected, rtol=1e-5, atol=1e-4)
    seconds = elapsed(lambda: batched(B))
    batch_payload = (W.shape[0] + W.shape[1]) * B * W.dtype.itemsize  # input + output of ONE batch, excluding W
    print(B, seconds, seconds/total*1e6, total/seconds, reference_time/seconds, batch_payload/1024)


Choose a batch from measured throughput, request-latency requirements and the memory budget.
B=1 is a one-column matrix product, distinct from the GEMV reference. The full dataset is resident;
its batch-payload model does not measure process peak RAM, and its timer excludes queueing.

## Optional — library routines and layout
Compare the same scalar sum computed by explicit compiled loops and an optimized library routine
on C- and F-contiguous arrays. Separately examine column reductions and mismatched copies.
There is no universal layout penalty or guarantee that a library hides every access cost.

In [ ]:
A_f = np.asfortranarray(A)
for label, arr in (("C", A), ("F", A_f)):
    reference = arr.sum()
    assert np.isclose(sum_i_outer(arr), reference)
    assert np.isclose(sum_j_outer(arr), reference)
    print(label, "compiled i outer:", ms(best_time(lambda: sum_i_outer(arr))),
          "compiled j outer:", ms(best_time(lambda: sum_j_outer(arr))),
          "NumPy total sum:", ms(best_time(lambda: arr.sum())))
# A separate operation: the column totals, with a vector output.
assert np.allclose(A.sum(axis=0), A_f.sum(axis=0))
print("axis=0 C:", ms(best_time(lambda: A.sum(axis=0))),
      "axis=0 F:", ms(best_time(lambda: A_f.sum(axis=0))))


In [ ]:
outC = np.empty((N, N))                      # destination: C-order

t_same = best_time(lambda: np.copyto(outC, A))    # C ← C : layouts agree
t_diff = best_time(lambda: np.copyto(outC, A_f))  # C ← F : different source/destination layouts
print("copyto  C ← C :", ms(t_same))
print("copyto  C ← F :", ms(t_diff))
print(f"ratio : {t_diff/t_same:.2f}×")
assert np.array_equal(outC, A)


## Optional — reuse output buffers
An output buffer avoids an allocation; the calculation still reads inputs and writes outputs.
NumPy out= calls need not fuse into one pass. Compare both time and allocations where available.

In [ ]:
buffer_n = 10_000_000
x = np.random.rand(buffer_n); y = np.random.rand(buffer_n); z = np.empty(buffer_n)

def out_of_place(): return 2*x + y             # allocates 80 MB every call
def in_place():                                 # writes into z, allocates nothing
    np.multiply(x, 2.0, out=z); np.add(z, y, out=z); return z

assert np.allclose(out_of_place(), in_place())
t_out = best_time(out_of_place, repeat=3)
t_in  = best_time(in_place,     repeat=3)
print("allocates :", ms(t_out))
print("in place  :", ms(t_in), f"-> {t_out/t_in:.1f}x, and an output allocation avoided")



## Optional — arithmetic intensity

In [ ]:
n = 1000
M = np.random.rand(n, n); v = np.random.rand(n); M2 = np.random.rand(n, n)
from threadpoolctl import threadpool_limits, threadpool_info
print(threadpool_info())
with threadpool_limits(limits=1, user_api="blas"):
    for label, fn, flops in (("matrix × vector", lambda: M @ v, 2*n*n),
                              ("matrix × matrix", lambda: M @ M2, 2*n**3)):
        t = best_time(fn)
        print(label, ms(t), "GFLOP/s", flops/t/1e9)


For Float64, an ideal GEMV reads roughly 8n² bytes for 2n² FLOP (~0.25 FLOP/byte). A square GEMM reading two inputs and writing one output has an ideal traffic lower bound of 24n² bytes: intensity ~n/12. Actual traffic depends on cache reuse and implementation; these models are not measurements of bandwidth.

## 4. GPU with PyTorch

The tensor's device selects CPU or CUDA execution. In Julia, methods specialized for
`CuArray` provide the GPU operations. Specify device, input dtype and accuracy before comparing.


In [ ]:
import torch
gpu = torch.cuda.is_available()
print("CUDA available :", gpu, "|", torch.cuda.get_device_name(0) if gpu else "CPU only")

def bench_torch(fn, gpu, repeat=5):
    if gpu: torch.cuda.synchronize()
    # warm-up
    fn();  torch.cuda.synchronize() if gpu else None
    import time
    best = float("inf")
    for _ in range(repeat):
        t0 = time.perf_counter(); fn()
        if gpu: torch.cuda.synchronize()
        best = min(best, time.perf_counter() - t0)
    return best

### Device and precision

`.cuda()` copies a CPU tensor to device memory and preserves its dtype. Choose `float32`
explicitly below. NumPy's random arrays default to `float64`; PyTorch defaults to `float32`.

Julia's `CuArray(A)` preserves the dtype, whereas the convenience function `cu(A)` can
narrow `Float64` to `Float32`. State any change in precision.


In [ ]:
a_np = np.random.rand(3, 3)          # numpy: float64
t_cpu = torch.rand(3, 3, dtype=torch.float32)             # torch: float32 by DEFAULT — not the same dtype!
print("numpy dtype :", a_np.dtype)
print("torch dtype :", t_cpu.dtype, "  <- explicitly chosen")
print("torch device:", t_cpu.device)

if gpu:
    t_gpu = t_cpu.cuda()             # THE COPY: host -> device
    print("after .cuda():", t_gpu.device, "|", type(t_gpu).__name__)
    print("same object? ", t_gpu is t_cpu, " | same values?", torch.equal(t_cpu, t_gpu.cpu()))
    # Operations require compatible devices.
    try:
        t_cpu + t_gpu
    except RuntimeError as e:
        print("cpu_tensor + gpu_tensor ->", str(e).split(",")[0])
    # .item() synchronizes and returns a scalar to the host:
    print("t_gpu.sum().item() =", round(t_gpu.sum().item(), 4), " <- this SYNCS and copies back")

### What the copy costs

Measure uploads, downloads and resident computation separately. A complete matrix product
with host inputs and output needs two uploads and one download. Measure that complete path
below; adding separate minimum times only gives an estimate.


In [ ]:
if gpu:
    K = 4096
    Ah = torch.rand(K, K, dtype=torch.float32)                                    # 64 MiB, float32, on the host
    Ad = Ah.cuda(); Bd = torch.rand(K, K, device="cuda")
    t_up   = bench_torch(lambda: Ah.cuda(),   gpu=True)      # host -> device
    t_down = bench_torch(lambda: Ad.cpu(),    gpu=True)      # device -> host
    t_mul  = bench_torch(lambda: Ad @ Bd,     gpu=True)      # the actual work
    print("host -> device :", ms(t_up))
    print("device -> host :", ms(t_down))
    print("matmul on GPU  :", ms(t_mul))
    print("Derived full-path estimate (not a measured job):", ms(2*t_up + t_mul + t_down))
else:
    print("(no GPU)")

### GPU output buffers (optional)
Keep intermediate arrays on the device and return the required result. Reusing an output may
avoid allocation; compare the measured time because pooling and kernel choices also matter.

In [ ]:
# Optional GPU output-buffer comparison; independent of the CPU reference cells.
if gpu:
    buffer_n = 10_000_000
    xg = torch.rand(buffer_n, device="cuda"); yg = torch.rand(buffer_n, device="cuda")
    zg = torch.empty(buffer_n, device="cuda")
    t_og = bench_torch(lambda: 2*xg + yg, gpu=True)
    t_ig = bench_torch(lambda: torch.add(yg, xg, alpha=2, out=zg), gpu=True)
    print("GPU new output:", ms(t_og), "reused output:", ms(t_ig))


In [ ]:
# Same Float32 inputs: CPU result, resident GPU work, complete host-to-host job.
def matmul_to_host(A, B):
    return (A.cuda() @ B.cuda()).cpu()

M = 4096
Ac = torch.rand(M, M, dtype=torch.float32)
Bc = torch.rand(M, M, dtype=torch.float32)
reference = Ac @ Bc
print("CPU, host result:", ms(bench_torch(lambda: Ac @ Bc, gpu=False)))
if gpu:
    Ag = Ac.cuda(); Bg = Bc.cuda()
    result = matmul_to_host(Ac, Bc)
    # Example application requirement; justify the tolerance for your actual problem.
    rtol = 1e-4
    relative_error = torch.linalg.norm(result-reference) / torch.linalg.norm(reference)
    print("Relative error:", relative_error.item(), "meets norm tolerance:", relative_error <= rtol)
    print("GPU, resident inputs/result:", ms(bench_torch(lambda: Ag @ Bg, gpu=True)))
    print("GPU, complete host-to-host job:", ms(bench_torch(lambda: matmul_to_host(Ac, Bc), gpu=True)))


In [ ]:
# Batching CPU vs GPU: read the GPU TOTAL column, not just the speedup
if gpu:
    nb = 4096
    Wt = torch.rand(nb, nb, device="cuda")       # 4096² GPU workload
    Wc = Wt.cpu()
    print("batch B | GPU total (ms) | GPU µs/ex | GPU GFLOP/s | CPU µs/ex | CPU/GPU")
    print("-"*74)
    for B in (1, 8, 64, 256, 1024):
        Xc = torch.rand(nb, B); Xg = Xc.cuda()
        tc = bench_torch(lambda: Wc @ Xc, gpu=False)
        tg = bench_torch(lambda: Wt @ Xg, gpu=True)
        print(f"{B:6d}  | {tg*1e3:14.3f} | {tg/B*1e6:9.2f} | "
              f"{2.0*nb*nb*B/tg/1e9:11.0f} | {tc/B*1e6:9.1f} | {tc/tg:6.1f}×")
else:
    print("(no GPU: enable the GPU accelerator in Colab)")

In [ ]:
# Monte-Carlo π: generate, count and reduce on the chosen device
def pi_torch(n, device):
    x = torch.rand(n, device=device, dtype=torch.float32); y = torch.rand(n, device=device, dtype=torch.float32)
    return 4.0 * torch.count_nonzero(x*x + y*y <= 1.0).item() / n

n_mc = 100_000_000
print("π (reference) :", np.pi)
print("π torch CPU :", pi_torch(n_mc, "cpu"))
if gpu:
    print("π torch GPU :", pi_torch(n_mc, "cuda"))
    print("CPU :", ms(bench_torch(lambda: pi_torch(n_mc, "cpu"), gpu=False)))
    print("GPU :", ms(bench_torch(lambda: pi_torch(n_mc, "cuda"), gpu=True)))
print("Approximate standard error:", 1.64 / n_mc**0.5)
# This is not an absolute-error guarantee; choose n for the required statistical accuracy.
# Timed calls include RNG, allocation, reduction, synchronization and scalar return.


## GPU decision

1. Is there enough parallel work?
2. Can intermediate data stay on the GPU?
3. Is the complete job faster at the required accuracy?

Record the timed scope, input dtype, numerical tolerance and statistical target where relevant.
Resident products and complete host-to-host products answer different questions. Compare the
required outputs and report your own CPU/BLAS configuration and device.
